# 12 — Tabel paper (CPU)

Yang mahal — mengubah citra jadi logit — sudah selesai di notebook 07, 10, dan 11.
Notebook ini membaca cache itu dan menghasilkan tabelnya. **Tanpa GPU.**

| Fase | Menjawab | Sumber |
|---|---|---|
| **1. Fungsi skor** THR/APS/RAPS/SAPS | apakah koreksinya bertahan di skor lain — δ_y didefinisikan di atas distribusi skor | dump CCC |
| **2. α** 0,01/0,05/0,10 | di mana batasnya | dump CCC |
| **3. Pesaing terbit** | **uji kebaruan sesungguhnya** — fuzzy classwise satu-satunya metode terbit yang terdefinisi di `n_y = 0` | repo LTC |
| **4. ImageNet-C** | apakah mekanismenya bertahan di bawah pergeseran distribusi | cache nb10 |
| **5. Backbone** ResNet-50/ViT-B/ConvNeXt-T | apakah klaimnya bergantung satu model | cache nb11 |

**Fase 3 yang paling menentukan.** Bandwidth fuzzy disapu dan yang **terbaik**
diambil — pesaingnya di-tune oracle, arah yang konservatif bagi kita. Biayanya
**terukur** ~25 menit per run (15 kandidat × ~99 s), jadi ia hanya jalan di satu
konfigurasi primer, bukan di setiap sel.

**Bisa dilanjutkan per run.** Setiap konfigurasi menulis laporannya ke
`runs/nb12_cache/` di Drive **segera setelah selesai**, dan run berikutnya
melewatinya. Sebuah kernel restart karena itu kehilangan **satu** konfigurasi, bukan
seluruh jam yang sudah dijalani — pelajaran dari crash 17 Agustus, di mana notebook
10 punya resume dan notebook ini tidak.

## 1. Config repo, unduhan, dan Drive

In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

# SUMBER DUMP. LTC sudah punya ID gdown konkret (dari notebook 00) -> nol langkah
# manual. CCC belum: ID-nya harus dibaca dari download_data.sh mereka. Jadi survei
# LTC dulu; pindah ke CCC hanya kalau tidak ada dump LTC yang memenuhi premis.
# TIDAK ADA saklar sumber. Versi sebelumnya punya SOURCE yang harus diedit manual,
# dan default-nya ('ltc') sudah diketahui GAGAL premis — jadi setiap run default
# berakhir dengan assert. Sekarang SEMUA sumber disurvei dalam satu jalan dan yang
# terbaik dipakai. Tabel perbandingannya sendiri adalah temuan yang dicari.
CCC_DATASETS = ('imagenet',)   # tambah 'inaturalist' (iNat-2021, 633 kelas) bila perlu
GID_SCORES_LTC = {'plantnet': '1k_PPQV3VJT44hz02CcnbqPstjQo70vGr',
                  'inaturalist': '1W8R8Jj2bhS2PbR-3X9vEw-WkanbOk6mq'}
# ID CCC dari download_data.sh mereka. Ditanam supaya TIDAK ada langkah manual:
# versi sebelumnya hanya mencetak skripnya dan menyuruh menjalankan gdown sendiri,
# yang membingungkan dan tidak perlu begitu ID-nya diketahui.
GID_SCORES_CCC = {'imagenet':   '1AQjUn3m010N_i6-sfD690W7mZq2RTwJz',   # 4,62 GB
                  'inaturalist':'1BUlQZhS_5x2LJpyxCGI1IkmRrkvmRD88',   # 6,72 GB (iNat-2021, 633 kelas)
                  'places365':  '119k7PE1l72fg5Rpez5brIOn28BwClqv2',   # 0,54 GB (365 kelas)
                  'cifar-100':  '1yXD9XqBxEJnJxcfnnduNK6nHHUU3iX_6'}   # 0,01 GB (100 kelas)
# Catatan daya uji: premis butuh >=500 kelas layak, jadi places365 (365 kelas) dan
# cifar-100 (100 kelas) TIDAK BISA memenuhinya secara konstruksi, berapa pun
# sampel per kelasnya. Yang mungkin: imagenet (1.000) dan iNat-2021 (633).
LTC_DATASETS = ('inaturalist', 'plantnet')   # rilis memuat varian -trunc juga
LOSS_VARIANT = 'cross_entropy'  # 'cross_entropy' | 'focal'. LTC mengirim SEMBILAN
                                # berkas dengan NAMA IDENTIK di subdirektori berbeda;
                                # tercampur = skor dari model lain, akurasi mirip,
                                # kalibrasi beda total. Notebook 00 kena isu yang sama.
# places365 (365 kelas) dan cifar-100 (100 kelas) TIDAK BISA memenuhi premis >=500
# kelas secara konstruksi, berapa pun sampel per kelasnya — jadi tidak diunduh.
N_CLASSES_EXPECTED = None      # None = jangan dipaksakan; dibaca dari dump

# --- PRIMER, ditetapkan di prereg_imagenet_gate.md. JANGAN diubah setelah melihat hasil.
ALPHA_PRIMARY = 0.10
N_CAL_PRIMARY = 25
N_BOOT_CLASS  = 400           # bootstrap tingkat-kelas untuk gate B
N_PERM_CLASS  = 1000          # permutasi tingkat-kelas untuk gate C (p_min = 1/1001)
STABLE_THRESHOLD = 0.90

# --- SEKUNDER
ALPHAS_SECONDARY = (0.01, 0.05)
N_CAL_SECONDARY  = 50
N_SPLITS_BC = 100
N_SPLITS_A  = 100
RUN_CLUSTERED_CP = True       # reproduksi baseline pada skor yang sama

FRAC_DESC, FRAC_CAL = 0.40, 0.30   # sisanya EVAL

# ANGGARAN BARIS. Dump ImageNet CCC nyata adalah (1.153.051 x 1.000) float32 = 4,61 GB
# -- sepuluh kali lebih besar dari yang tercatat di release_audit.md. Memuatnya penuh
# lalu membuat salinan turunan (thr_lac, entropi, np.partition) melewati RAM Colab.
#
# Subsampling di sini BUKAN perubahan kriteria: premis butuh >=84 sampel/kelas dan
# anggaran ini menyisakan ~230/kelas. Ia diambil sebagai FRAKSI per kelas, bukan cap
# tetap, karena cap tetap membuat semua hitungan kelas SAMA -> log_prevalence konstan
# -> ablasi prevalensi jadi hampa. Fraksi mempertahankan struktur prevalensinya.
MAX_ROWS = 250_000            # None = pakai seluruh dump
SEED = 42
# =======================================================================
print(f'PRIMER: alpha={ALPHA_PRIMARY} n_cal={N_CAL_PRIMARY} '
      f'n_boot={N_BOOT_CLASS} n_perm={N_PERM_CLASS}')


## 2. Mount + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
subprocess.run(['pip','install','-q','gdown'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Siapkan SEMUA dump — TANPA citra, TANPA GPU

Dump LTC dipakai lokasi yang sama dengan notebook 00 (`released_scores/<dataset>`), jadi kalau
sudah ada tidak diunduh ulang. Dump CCC diunduh otomatis dengan ID yang sudah ditanam.

Keduanya disurvei berdampingan di sel 4. Itu bukan pemborosan: **perbandingan ekor-panjang
versus berimbang adalah temuannya**, dan menurunkannya dari satu tabel lebih kuat daripada dari
dua run terpisah.


In [ ]:
import glob, zipfile, numpy as np

def ltc_dir(ds):
    return f'{DRIVE_ROOT}/released_scores/{ds}'

def ccc_dir(ds):
    return f'{DRIVE_ROOT}/scores_ccc/{ds}'

for ds in LTC_DATASETS:
    d = ltc_dir(ds); os.makedirs(d, exist_ok=True)
    if glob.glob(f'{d}/**/*_softmax.npy', recursive=True):
        print(f'ltc/{ds}: sudah ada, dilewati')
        continue
    print(f'ltc/{ds}: mengunduh...')
    z = f'{d}/{ds}.zip'
    r = subprocess.run(['gdown', GID_SCORES_LTC[ds], '-O', z], capture_output=True, text=True)
    if r.returncode:
        print('  gdown gagal:', r.stderr.strip()[:300])
    else:
        subprocess.run(['unzip','-o','-q',z,'-d',d], check=False)

def inventory(d, label):
    files = [p for p in sorted(glob.glob(f'{d}/**/*', recursive=True)) if os.path.isfile(p)]
    print(f'  isi {label}: {len(files)} berkas')
    for p in files[:25]:
        print(f'    {os.path.relpath(p, d):56s} {os.path.getsize(p)/1e6:9.2f} MB')
    return files

def looks_like_html(p):
    # Kegagalan kuota Google Drive menulis halaman HTML DAN mengembalikan kode 0.
    # Inilah sebabnya returncode tidak boleh dipercaya sebagai bukti unduhan berhasil.
    try:
        with open(p, 'rb') as fh:
            head = fh.read(400)
    except OSError:
        return False, b''
    low = head.lower()
    return (b'<html' in low or b'<!doctype html' in low), head

for ds in CCC_DATASETS:
    d = ccc_dir(ds); os.makedirs(d, exist_ok=True)
    mat = f'/content/ccc_npy/{ds}'
    # Tiga keadaan, dan versi sebelumnya hanya mengenali yang pertama:
    #   (a) .npy sudah dimaterialkan di /content -> tidak ada kerja
    #   (b) .npz ada di Drive tapi .npy hilang (sesi baru; /content ephemeral)
    #       -> ekstrak ulang, JANGAN unduh 4,6 GB lagi
    #   (c) tidak ada apa pun -> unduh
    # Pemeriksaan lama hanya mencari .npy DI DRIVE, yang tidak pernah ada karena
    # materialisasinya ke /content. Jadi setiap sesi baru mengunduh ulang 4,6 GB.
    if glob.glob(f'{mat}/*.npy') or glob.glob(f'{d}/**/*.npy', recursive=True):
        print(f'ccc/{ds}: .npy sudah ada, dilewati')
        continue
    if glob.glob(f'{d}/*.npz'):
        print(f'ccc/{ds}: .npz ada di Drive, ekstrak ulang tanpa mengunduh')
    else:
        print(f'ccc/{ds}: mengunduh (beberapa GB, sabar)...')
    if not glob.glob(f'{d}/*.npz'):
        r = subprocess.run(['gdown', '--fuzzy', GID_SCORES_CCC[ds]],
                           cwd=d, capture_output=True, text=True)
        if r.stdout.strip():
            print('  stdout:', r.stdout.strip()[-500:])
        if r.stderr.strip():
            print('  stderr:', r.stderr.strip()[-300:])
        print(f'  returncode: {r.returncode}  <- BUKAN bukti; diverifikasi di bawah')

    for p in sorted(glob.glob(f'{d}/*')):
        low = p.lower()
        if low.endswith('.zip'):
            subprocess.run(['unzip','-o','-q',p,'-d',d], check=False)
        elif low.endswith(('.tar.gz','.tgz','.tar')):
            subprocess.run(['tar','-xf',p,'-C',d], check=False)

    # .npz ADALAH zip berisi beberapa .npy. Versi sebelumnya mencocokkan ekstensi
    # secara literal ('.zip'/'.tar'), jadi imagenet.npz dilewati dan tidak ada .npy
    # terbentuk -- padahal unduhannya berhasil penuh. Satu baris, kegagalan bisu.
    #
    # Header .npy dibaca lewat zipfile TANPA mendekompresi isinya, supaya bentuk dan
    # dtype tiap anggota terlihat tanpa memuat gigabyte. Lalu HANYA dua array yang
    # dibutuhkan dimaterialkan, dan ke /content (ephemeral) bukan Drive -- mengekstrak
    # seluruh 4,6 GB ke Drive akan menggandakan pemakaian kuota tanpa alasan.
    for p in sorted(glob.glob(f'{d}/*.npz')):
        print(f'  membaca header {os.path.basename(p)} (tanpa dekompresi)...')
        members = []
        with zipfile.ZipFile(p) as zf:
            for nm in zf.namelist():
                try:
                    with zf.open(nm) as fh:
                        ver = np.lib.format.read_magic(fh)
                        if ver == (1, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_1_0(fh)
                        elif ver == (2, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_2_0(fh)
                        else:
                            continue
                    members.append((nm, shp, dt))
                    print(f'    {nm:36s} {str(shp):20s} {dt}')
                except Exception as e:
                    print(f'    {nm:36s} header tak terbaca: {type(e).__name__}')

        twod = [m for m in members if len(m[1]) == 2]
        oned = [m for m in members if len(m[1]) == 1]
        if not twod or not oned:
            print('  npz ini tidak memuat pasangan (2-D, 1-D) — kirim daftar di atas.')
            continue
        sc = max(twod, key=lambda m: m[1][0] * m[1][1])
        lb = next((m for m in oned if m[1][0] == sc[1][0]), None)
        if lb is None:
            print(f'  tidak ada array 1-D sepanjang {sc[1][0]} untuk mendampingi {sc[0]}')
            continue
        out = f'/content/ccc_npy/{ds}'
        os.makedirs(out, exist_ok=True)
        print(f'  materialkan {sc[0]} {sc[1]} dan {lb[0]} {lb[1]} -> {out}')
        # zipfile.namelist() memberi nama DENGAN sufiks '.npy', tetapi NpzFile
        # diindeks TANPA sufiks -> z['softmax.npy'] KeyError, z['softmax'] benar.
        # Ditemukan oleh tes sintetik sebelum run nyata.
        k_sc = sc[0][:-4] if sc[0].endswith('.npy') else sc[0]
        k_lb = lb[0][:-4] if lb[0].endswith('.npy') else lb[0]
        with np.load(p, allow_pickle=False) as z:
            np.save(f'{out}/scores.npy', z[k_sc])
            np.save(f'{out}/labels.npy', z[k_lb])

    files = inventory(d, f'ccc/{ds}')
    npys = (glob.glob(f'{d}/**/*.npy', recursive=True)
            + glob.glob(f'/content/ccc_npy/{ds}/*.npy'))
    if not npys:
        print(f'  GAGAL: tidak ada .npy terbentuk untuk ccc/{ds}.')
        for p in files:
            is_html, head = looks_like_html(p)
            if is_html:
                print(f'  PENYEBAB: {os.path.basename(p)} adalah HALAMAN HTML, bukan data.')
                print('  Itu batas kuota Google Drive; gdown tetap keluar dengan kode 0.')
                print('  cuplikan:', ' '.join(head.decode('utf-8','replace').split())[:300])
                print(f'  URL: https://drive.google.com/uc?id={GID_SCORES_CCC[ds]}')
                break
        else:
            print('  Berkas ADA tetapi tidak menghasilkan .npy — kirim daftar di atas.')
    else:
        print(f'  OK: {len(npys)} .npy siap dipakai')

# dua akar untuk CCC: Drive (kalau .npy langsung) dan /content (hasil materialisasi
# anggota .npz). Keduanya diperiksa supaya tidak peduli bentuk rilisnya.
roots = ([(ltc_dir(ds), 'ltc', ds) for ds in LTC_DATASETS]
         + [(ccc_dir(ds), 'ccc', ds) for ds in CCC_DATASETS]
         + [(f'/content/ccc_npy/{ds}', 'ccc', ds) for ds in CCC_DATASETS])
found = []
for rt, src, ds in roots:
    for f in sorted(glob.glob(f'{rt}/**/*.npy', recursive=True)):
        found.append((f, src, ds))
print()
print(f'total .npy: {len(found)}')
for f, src, ds in found[:60]:
    a = np.load(f, mmap_mode='r')
    print(f'  [{src}] {os.path.basename(f):52s} {str(a.shape):18s} {a.dtype}')
if not found:
    print('TIDAK ADA .npy sama sekali. Isi direktori mentah:')
    for rt, src, ds in roots:
        for p in sorted(glob.glob(f'{rt}/**/*', recursive=True))[:25]:
            if os.path.isfile(p):
                print(f'  {os.path.relpath(p, DRIVE_ROOT):64s} {os.path.getsize(p)/1e6:8.1f} MB')


### 3b. Referensi — dari mana ID CCC berasal (opsional, tidak perlu dijalankan)

ID di sel 1 diambil dari `download_data.sh` milik CCC. Sel ini hanya untuk memverifikasi
bahwa ID-nya belum berubah; ia **tidak diperlukan** untuk menjalankan notebook.


In [ ]:
SHOW_CCC_SCRIPT = False
if SHOW_CCC_SCRIPT:
    if not os.path.isdir('/content/ccc'):
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/tiffanyding/class-conditional-conformal.git',
                        '/content/ccc'], check=False)
    print(open('/content/ccc/download_data.sh').read())
    print('bandingkan dengan GID_SCORES_CCC di sel 1')
else:
    print('dilewati (ID sudah ditanam di sel 1)')


## 4. Dump, kepala, dan repo pesaing

In [ ]:
import glob

def _pair(scores):
    for suf in ('_softmax.npy', '_scores.npy', 'scores.npy'):
        if scores.endswith(suf):
            cand = scores[: -len(suf)] + suf.replace('softmax', 'labels').replace(
                'scores', 'labels')
            if os.path.exists(cand):
                return cand
    cand = os.path.join(os.path.dirname(scores), 'labels.npy')
    return cand if os.path.exists(cand) else None

DUMPS = {}

# CCC: materialisasi sel di atas menaruhnya di /content/ccc_npy/<ds>/
for p in sorted(glob.glob('/content/ccc_npy/*/scores.npy')):
    ds = os.path.basename(os.path.dirname(p))
    lab = _pair(p)
    if lab:
        DUMPS['ccc_' + ds] = {'scores': p, 'labels': lab, 'eval_scores': None,
                              'eval_labels': None, 'max_rows': MAX_ROWS}

# LTC: pasangkan cal (DESC+CAL) dengan test (EVAL penuh)
for ds in LTC_DATASETS:
    d = f'{DRIVE_ROOT}/released_scores/{ds}'
    cal = sorted(glob.glob(f'{d}/**/*cal_softmax.npy', recursive=True))
    tst = sorted(glob.glob(f'{d}/**/*test_softmax.npy', recursive=True))
    cal = [p for p in cal if LOSS_VARIANT in p or LOSS_VARIANT == 'cross_entropy']
    tst = [p for p in tst if LOSS_VARIANT in p or LOSS_VARIANT == 'cross_entropy']
    if cal and tst and _pair(cal[0]) and _pair(tst[0]):
        DUMPS['ltc_' + ds] = {'scores': cal[0], 'labels': _pair(cal[0]),
                              'eval_scores': tst[0], 'eval_labels': _pair(tst[0]),
                              'max_rows': None}

assert DUMPS, 'tidak ada dump siap pakai -- periksa sel penyiapan di atas'
for k, v in DUMPS.items():
    a = np.load(v['scores'], mmap_mode='r')
    line = '  ' + k.ljust(18) + ' cal ' + str(a.shape)
    if v['eval_scores']:
        e = np.load(v['eval_scores'], mmap_mode='r')
        line += '  eval ' + str(e.shape) + '  (dump terpisah)'
    else:
        line += '  eval = 30% dari dump yang sama'
    print(line)

In [ ]:
import numpy as np, os, glob, subprocess

HEAD_DIR = '/content/head'
os.makedirs(HEAD_DIR, exist_ok=True)
HEADS = {}          # K -> (path W, path b)

def _register(tag, W, b):
    w_p = f'{HEAD_DIR}/{tag}_fc_weight.npy'
    b_p = f'{HEAD_DIR}/{tag}_fc_bias.npy'
    np.save(w_p, W)
    np.save(b_p, np.zeros(len(W)) if b is None else b)
    HEADS[int(W.shape[0])] = (w_p, b_p)
    print('  kepala', tag, W.shape, '-> K =', W.shape[0])

# --- 1. torchvision ResNet-50 (ImageNet-1k). Model tersupervisi yang BERBEDA dari
# SimCLRv2+probe penghasil skor CCC: ketidakcocokan itu justru yang membuat phi
# eksogen sepenuhnya terhadap delta_y.
tv_w = f'{HEAD_DIR}/torchvision_imagenet_fc_weight.npy'
if os.path.exists(tv_w):
    W = np.load(tv_w)
    HEADS[int(W.shape[0])] = (tv_w, tv_w.replace('_weight', '_bias'))
    print('  kepala torchvision_imagenet sudah ada -> K =', W.shape[0])
else:
    from pcc.descriptors.head_weights import load_torchvision_resnet50_head
    W, b = load_torchvision_resnet50_head()
    _register('torchvision_imagenet', W, b)

# --- 2. Kepala LTC untuk Pl@ntNet dan iNat-2018. Run pertama Phase 2 gagal di kedua
# dataset itu, tetapi HANYA keluarga ruang-output yang pernah dijalankan di sana --
# dan keluarga itu juga gagal di ImageNet. Jadi itu kegagalan KELUARGA phi, bukan
# kegagalan dataset, dan checkpoint yang dirilis membuatnya bisa diuji tanpa GPU.
GID_MODELS = '1tS-M-4IYyCGMeIxxyrgx2-XCZgdvw18S'   # models.zip, dari notebook 00
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints/ltc_models'
os.makedirs(CKPT_DIR, exist_ok=True)
if not glob.glob(f'{CKPT_DIR}/**/*model*.pth', recursive=True):
    print('mengunduh models.zip LTC (6 ResNet-50)...')
    subprocess.run(['gdown', GID_MODELS, '-O', f'{CKPT_DIR}/models.zip'], check=True)
    subprocess.run(['unzip', '-o', f'{CKPT_DIR}/models.zip', '-d', CKPT_DIR],
                   check=True)

def _variant_ok(path):
    # PERANGKAP dari notebook 00: LTC mengirim ENAM model dengan NAMA BERKAS
    # IDENTIK dan menaruh varian focal di subdirektori 'focal_loss'. Glob rekursif
    # bisa mengambil mana saja, jadi checkpoint dan skor bisa diam-diam berasal dari
    # varian berbeda -- akurasi mirip, kalibrasi beda total.
    is_focal = 'focal' in path.replace(chr(92), '/').lower()
    return is_focal if LOSS_VARIANT == 'focal' else (not is_focal)

from pcc.data.ltc_datasets import NUM_CLASSES
for ds in LTC_DATASETS:
    tag = 'ltc_' + ds
    w_p = f'{HEAD_DIR}/{tag}_fc_weight.npy'
    if os.path.exists(w_p):
        W = np.load(w_p, mmap_mode='r')
        HEADS[int(W.shape[0])] = (w_p, w_p.replace('_weight', '_bias'))
        print('  kepala', tag, 'sudah ada -> K =', W.shape[0])
        continue
    cands = sorted(glob.glob(f'{CKPT_DIR}/**/best-{ds}-model.pth', recursive=True))
    keep = [p for p in cands if _variant_ok(p)]
    print('  checkpoint', ds, ':', len(cands), 'kandidat,', len(keep),
          'cocok varian', LOSS_VARIANT)
    for p in cands:
        print('     ' + ('* ' if _variant_ok(p) else '  ') + p)
    if not keep:
        print('     DILEWATI: tidak ada checkpoint varian', LOSS_VARIANT)
        continue
    try:
        from pcc.extract.backbones import load_ltc_resnet50
        m = load_ltc_resnet50(keep[0], NUM_CLASSES[ds], None)
        W = m.fc.weight.detach().cpu().numpy()
        b = m.fc.bias.detach().cpu().numpy() if m.fc.bias is not None else None
        del m
        assert W.shape[0] == NUM_CLASSES[ds], (W.shape, NUM_CLASSES[ds])
        _register(tag, W, b)
    except Exception as e:
        print('     GAGAL memuat:', type(e).__name__, str(e)[:160])

print()
print('kepala tersedia per jumlah kelas:', {k: os.path.basename(v[0])
                                            for k, v in sorted(HEADS.items())})

### 4b. Repo LTC — sumber pesaing terbit

Metodenya dipanggil dari rilis penulisnya, tidak ditulis ulang. `fuzzy_classwise_CP`
memakai `np.quantile(..., weights=)` yang baru ada di **numpy 2.0**, jadi versinya
diperiksa di sini — kalau tidak, kegagalannya muncul 25 menit kemudian sebagai
`TypeError` di tengah fase 3.

In [ ]:
LTC_URL = 'https://github.com/tiffanyding/long-tail-conformal.git'
LTC_DIR = '/content/ltc'
if not (os.path.isdir(LTC_DIR) and os.listdir(LTC_DIR)):
    subprocess.run(['git', 'clone', '--depth', '1', LTC_URL, LTC_DIR], check=True)
assert os.path.exists(LTC_DIR + '/utils/conformal_utils.py'), 'repo LTC tak lengkap'
subprocess.run(['pip', 'install', '-q', 'scikit-learn', 'matplotlib'], check=False)

import numpy as np
_v = tuple(int(x) for x in np.__version__.split('.')[:2])
print('numpy', np.__version__)
if _v < (2, 0):
    print('PERINGATAN: fuzzy_classwise_CP butuh np.quantile(weights=) dari numpy',
          '2.0. Fase 3 akan mencatat kegagalannya, bukan diam-diam melewatkannya.')
else:
    print('numpy cukup baru untuk fuzzy_classwise_CP')
print('repo pesaing:', LTC_DIR)

## 5. Penyimpan Drive

In [ ]:
import shutil, time
RUN_DIR = DRIVE_ROOT + '/runs/nb12_' + time.strftime('%Y%m%d_%H%M%S')
os.makedirs(RUN_DIR, exist_ok=True)
# CACHE_DIR sengaja TANPA stempel waktu: itu satu-satunya cara run berikutnya
# menemukan pekerjaan yang sudah selesai. RUN_DIR yang berstempel tetap ada
# sebagai snapshot per-run.
CACHE_DIR = DRIVE_ROOT + '/runs/nb12_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
_done = sorted(glob.glob(CACHE_DIR + '/nb12_*.json'))
print('tujuan:', RUN_DIR)
print('cache resume:', CACHE_DIR, '->', len(_done), 'run sudah selesai')

def save_to_drive(tag):
    n_ok = n_bad = 0
    for p in sorted(glob.glob('pcc/reports/*.json')):
        d = os.path.join(RUN_DIR, os.path.basename(p))
        try:
            if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(p):
                n_ok += 1
                continue
            shutil.copy2(p, d)
            ok = os.path.getsize(d) == os.path.getsize(p)
            n_ok += int(ok); n_bad += int(not ok)
        except Exception as e:
            n_bad += 1
            print('   gagal', os.path.basename(p), e)
    print('   [{}] tersimpan {} gagal {} -> Drive'.format(tag, n_ok, n_bad))
    return n_bad == 0

save_to_drive('awal')

## 6. Konfigurasi eksperimen — EDIT ME kedua

In [ ]:
# === EDIT ME ===========================================================
ALPHA       = 0.10                     # alpha primer: tempat PCC lolos
ALPHAS      = (0.01, 0.05, 0.10)
SCORES      = ('thr', 'aps', 'raps', 'saps')
SEEDS_MAIN  = (0, 1, 2, 3, 4)
SEEDS_COMP  = (0, 1, 2)                # pesaing: ~25 mnt/run, jadi tiga
FRAC_CAL_MAIN = 0.70                   # 175 baris cal/kelas, 75 untuk EVAL
N_CAL_MAIN  = 25
FRAC_CAL_BB = 0.30                     # backbone: 50k baris, cal 15/kelas
IC_SEVERITIES = ()                     # () = semua; (5,) untuk memangkas
COMP_ON_IC  = False                    # pesaing di ImageNet-C: ~3 mnt x kondisi
# Fase 3 memakai dump yang DISUBSAMPEL. Pesaingnya yang boros: compute_ranks di
# dalam rc3p mengalokasi dua matriks int64 seukuran (n_cal x K) -- ~2 GB pada
# irisan penuh -- lalu berjalan dengan loop Python atas setiap baris. PCC dan
# pesaingnya melihat irisan yang SAMA, jadi perbandingannya tetap internally
# consistent; yang berubah hanya seberapa banyak baris kalibrasi keduanya dapat.
MAX_ROWS_COMP = 100_000
# =======================================================================

PRIMARY = 'ccc_imagenet'
assert PRIMARY in DUMPS, 'dump primer tak ada: ' + repr(sorted(DUMPS))
PRIMARY_S = DUMPS[PRIMARY]['scores']
PRIMARY_Y = DUMPS[PRIMARY]['labels']
_K = int(np.load(PRIMARY_S, mmap_mode='r').shape[1])
HEAD_W, HEAD_B = HEADS[_K]
print('dump primer', PRIMARY, '| K =', _K)
print('kepala:', os.path.basename(HEAD_W))

BACKBONES = {}
for d in sorted(glob.glob(DRIVE_ROOT + '/backbones/*')):
    if all(os.path.exists(d + '/' + f) for f in
           ('scores.npy', 'labels.npy', 'fc_weight.npy', 'fc_bias.npy')):
        BACKBONES[os.path.basename(d)] = d
print('backbone dari cache nb11:', sorted(BACKBONES) or 'TIDAK ADA')

n_runs = (len(SCORES) + len(ALPHAS)) * len(SEEDS_MAIN) + len(SEEDS_COMP) \
         + 3 * len(SEEDS_MAIN) * len(BACKBONES) // 3
print('perkiraan run non-ImageNet-C:', n_runs)

## 7. Runner bersama

`build()` memberi nilai pada **setiap** atribut, lalu penjaga `assert hasattr`
menolak nama yang tidak dikenal. Itu bukan hiasan: satu kali salah ketik nama
atribut pernah membunuh seluruh 48 konfigurasi notebook 08 karena diam-diam
dianggap default.

In [ ]:
import json, traceback, collections
from pcc.experiments import phase2_pcc as drv
from pcc.utils.io import write_report
from pcc.eval.stats import mean_ci, holm_bonferroni

RESULTS, FAILED = [], []

class A: pass

def build(**over):
    """Argumen driver. Setiap atribut diberi nilai di sini, jadi penjaga di bawah
    menangkap salah ketik nama alih-alih membiarkannya lewat sebagai default."""
    x = A()
    x.scores = x.labels = None
    x.eval_scores = x.eval_labels = None
    x.max_rows = None
    x.dataset = 'unset'
    x.reports_dir = 'pcc/reports'
    x.alpha, x.n_cal = 0.10, 25
    x.heldout_frac = 0.30
    x.frac_desc, x.frac_cal = 0.0, 0.70
    x.phi = 'head'
    x.head_weights = x.head_bias = None
    x.distance_holdout = 'w_cos_knn_1'
    x.stat = 'worst'
    x.ccc_root = LTC_DIR if os.path.isdir(LTC_DIR) else None
    x.competitors = False
    x.score = 'thr'
    x.cal_depth = None
    x.lam_override = None
    x.n_star_rule = 'oos'
    x.no_recalibrate = False
    x.feature_group = 'all'
    x.seed = 0
    x.name = None
    x.print_json = False
    for k, v in over.items():
        assert hasattr(x, k), 'atribut tak dikenal: ' + k   # salah ketik = diam
        setattr(x, k, v)
    assert x.scores and x.labels, 'scores/labels wajib'
    return x

def run_one(phase, tag, **over):
    """Satu konfigurasi, dan ia DILEWATI kalau laporannya sudah ada di Drive.

    Notebook 10 punya resume, notebook 12 tidak -- dan itu kesalahan yang mahal: satu
    kernel restart di fase 3 membuang seluruh fase 1 dan 2 juga. CACHE_DIR sengaja
    TIDAK berstempel waktu, supaya run berikutnya benar-benar menemukannya."""
    t0 = time.time()
    nm = 'nb12_{}_{}_s{}'.format(phase, tag, over.get('seed', 0)).replace('.', 'p')
    cached = os.path.join(CACHE_DIR, nm + '.json')
    if os.path.exists(cached) and os.path.getsize(cached) > 200:
        try:
            pay = json.load(open(cached))
            RESULTS.append(dict(phase=phase, tag=tag, seed=over.get('seed', 0),
                                secs=pay.get('runtime_seconds') or 0.0,
                                res=pay['results'], conclusion=pay['conclusion']))
            shutil.copy2(cached, os.path.join('pcc/reports', nm + '.json'))
            t2 = pay['results'].get('table_2_heldout')
            st = t2['primary_stat'] if t2 else None
            print('  {:34s} s{} CACHE  | T2 {:+.4f} | {}'.format(
                tag, over.get('seed', 0),
                t2['delta'].get(st, float('nan')) if t2 else float('nan'),
                pay['conclusion']), flush=True)
            return pay['results']
        except Exception as e:
            print('  cache rusak, dihitung ulang:', nm, e)
    try:
        x = build(**over)
        r = drv.run(x)
        c = drv.verdict(r, x.stat)
        pth = write_report(x.reports_dir, nm, hypothesis=drv.HYPOTHESIS,
                           pass_criteria=drv.PASS_CRITERIA, config=vars(x),
                           seed=x.seed, results=r, conclusion=c, started_at=t0)
        # ke cache Drive SEGERA, bukan di akhir fase: yang mahal di sini adalah
        # satu run, dan satu run yang hilang tidak boleh menarik yang lain
        try:
            shutil.copy2(str(pth), os.path.join(CACHE_DIR, nm + '.json'))
        except Exception as e:
            print('   gagal menulis cache:', e)
        RESULTS.append(dict(phase=phase, tag=tag, seed=x.seed,
                           secs=time.time() - t0, res=r, conclusion=c))
        t2 = r.get('table_2_heldout')
        st = t2['primary_stat'] if t2 else None
        print('  {:34s} s{} {:5.0f}s | lam {:.3f} | T2 {:+.4f} | {}'.format(
            tag, x.seed, time.time() - t0, r['pcc']['lambda'],
            t2['delta'].get(st, float('nan')) if t2 else float('nan'), c),
            flush=True)
        return r
    except Exception as e:
        FAILED.append(dict(phase=phase, tag=tag, error=type(e).__name__ + ': ' + str(e)))
        print('  {:34s} GAGAL: {}'.format(tag, str(e)[:140]), flush=True)
        traceback.print_exc()
        return None

def n_cal_for(cal_rows_per_class, wanted=(25, 15, 10, 5)):
    """n_cal terbesar yang MASIH tercapai pada irisan sedalam ini.

    n_cal adalah kriteria pra-registrasi, jadi driver menolak menurunkannya sendiri.
    Menurunkannya di sini eksplisit, dan nilai yang dibuang dicatat -- bukan diam-diam
    dipilih supaya run-nya lolos."""
    ok = [n for n in wanted if n <= cal_rows_per_class]
    return (ok[0] if ok else 1), [n for n in wanted if n > cal_rows_per_class]

print('runner siap | pesaing dari', LTC_DIR if os.path.isdir(LTC_DIR) else 'TIDAK ADA')

## 8. Cache ImageNet-C → dump softmax

Cache notebook 10 berisi **logit**, dan paper UM-TTA mengalibrasi lewat Platt
scaling. Jadi logit harus dibagi suhu **sebelum** softmax — softmax mentah bukan
skor mereka, dan tidak ada yang di bawah sini akan menyadari bedanya. Suhunya
dibaca dari manifest tiap seed, bukan diasumsikan 1,0.

In [ ]:
import json, torch

IC_ROOT = DRIVE_ROOT + '/umtta/imagenetc_resnet50'
IC_WORK = '/content/ic_npy'
os.makedirs(IC_WORK, exist_ok=True)

def load_pt(p):
    try:
        return torch.load(p, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(p, map_location='cpu')

def softmax_npy(pt_path, T, out_prefix):
    """logit -> softmax(logit / T) -> .npy, dengan suhu Platt dari manifest.

    Pembagian dengan T bukan hiasan: paper UM-TTA mengalibrasi lewat Platt scaling,
    jadi softmax mentah bukan skor MEREKA, dan tidak ada yang di bawah sini akan
    menyadari bedanya. Ini pelajaran dari notebook 08."""
    sp, lp = out_prefix + '_scores.npy', out_prefix + '_labels.npy'
    if os.path.exists(sp) and os.path.exists(lp):
        return sp, lp
    d = load_pt(pt_path)
    z = d['logits'].float() / float(T)
    p = torch.softmax(z, dim=1).numpy().astype('float32')
    np.save(sp, p)
    np.save(lp, d['labels'].numpy().astype('int64'))
    del d, z, p
    return sp, lp

IC_CASES = []
for man_p in sorted(glob.glob(IC_ROOT + '/*/seed_*/manifest.json')):
    man = json.load(open(man_p))
    sd = os.path.dirname(man_p)
    sub = os.path.basename(os.path.dirname(sd))
    seed = int(man['seed'])
    files = man.get('files', {})
    if 'baseline_cal' not in files or 'baseline_test' not in files:
        continue
    T = float(man['temperatures']['baseline'])
    IC_CASES.append(dict(sub=sub, seed=seed, T=T,
                         corruption=man['corruption'], severity=int(man['severity']),
                         cal=os.path.normpath(os.path.join(sd, files['baseline_cal'])),
                         test=os.path.normpath(os.path.join(sd, files['baseline_test'])),
                         cal_size=int(man['cal_size']),
                         test_size=int(man['test_size'])))

print('kondisi ImageNet-C ditemukan:', len(IC_CASES))
assert IC_CASES, 'cache notebook 10 tidak terbaca di ' + IC_ROOT
corrs = sorted({c['corruption'] for c in IC_CASES})
sevs = sorted({c['severity'] for c in IC_CASES})
seeds_ic = sorted({c['seed'] for c in IC_CASES})
print('  {} korupsi x {} severity x {} seed'.format(len(corrs), len(sevs), len(seeds_ic)))
print('  korupsi:', ', '.join(corrs))
print('  suhu Platt: {:.4f} .. {:.4f}'.format(
    min(c['T'] for c in IC_CASES), max(c['T'] for c in IC_CASES)))
assert len(corrs) * len(sevs) * len(seeds_ic) == len(IC_CASES), 'grid tidak lengkap'

## FASE 1 — sapuan fungsi skor

δ_y didefinisikan **di atas distribusi skor**, jadi "apakah koreksinya bertahan
dengan skor lain" adalah pertanyaan orde pertama tentang metodenya. Sampai baru-baru
ini driver mengunci THR/LAC, sehingga setiap angka PCC memakai satu skor saja.

APS dan RAPS diverifikasi elementwise lawan implementasi di rilis LTC; SAPS ditulis
dari papernya (tidak ada rujukan di rilis itu) dan dikunci oleh properti.

In [ ]:
print('FASE 1: sapuan fungsi skor di dump primer |', SCORES, '| seed', SEEDS_MAIN)
print('phi kepala, alpha', ALPHA, ', frac_cal', FRAC_CAL_MAIN)
for sc in SCORES:
    for s in SEEDS_MAIN:
        run_one('S_score', 'score_' + sc, scores=PRIMARY_S, labels=PRIMARY_Y,
                dataset='ccc_imagenet', max_rows=MAX_ROWS, score=sc, seed=s,
                alpha=ALPHA, frac_cal=FRAC_CAL_MAIN, n_cal=N_CAL_MAIN,
                head_weights=HEAD_W, head_bias=HEAD_B)
save_to_drive('fase 1 skor')

## FASE 2 — sapuan α

In [ ]:
print('FASE 2: sapuan alpha |', ALPHAS, '| seed', SEEDS_MAIN)
for a in ALPHAS:
    for s in SEEDS_MAIN:
        run_one('S_alpha', 'alpha{}'.format(a), scores=PRIMARY_S, labels=PRIMARY_Y,
                dataset='ccc_imagenet', max_rows=MAX_ROWS, alpha=a, seed=s,
                frac_cal=FRAC_CAL_MAIN, n_cal=N_CAL_MAIN,
                head_weights=HEAD_W, head_bias=HEAD_B)
save_to_drive('fase 2 alpha')

## FASE 3 — pesaing terbit (paling menentukan)

`fallback_policy.md` dibekukan **24 Juli 2026, sebelum eksperimen apa pun**, dan
memprediksi apa yang setiap metode lakukan pada kelas ber-`n_y = 0`. Diverifikasi
lawan sumber rilisnya:

| metode | pada `n_y = 0` |
|---|---|
| classwise CP | `+inf` — set = seluruh ruang label |
| RC3P | `+inf`, dan ia memperingatkan sendiri |
| clustered CP | satu ambang bersama (fallback marginalnya) |
| fuzzy-`quantile` | terdefinisi, tapi semua kelas held-out runtuh ke satu titik |
| fuzzy-`random` | berbeda per kelas, dari embedding **acak** |
| fuzzy-`rarity` | berbeda per kelas dari prevalensi — **informatif** |

Semuanya dibandingkan sebagai **vektor ambang per-kelas** lewat mesin evaluasi yang
sama dengan PCC: ukuran set dicocokkan, statistik sama, plafon sama.

In [ ]:
# Pesaing terbit. Biayanya TERUKUR ~25 menit per run di skala ini (15 kandidat fuzzy
# x ~99 s), jadi ia dijalankan di satu konfigurasi utama dengan sedikit seed, bukan di
# setiap sel. Itu juga tempat yang benar untuk tabel baseline: satu perbandingan di
# setting primer, bukan pengulangan yang sama di seluruh grid.
print('FASE 3: pesaing terbit, {} seed | dump disubsampel ke {} baris'.format(
    len(SEEDS_COMP), MAX_ROWS_COMP))
# MAX_ROWS_COMP = None berarti 'pakai seluruh dump', jadi jangan mengalikannya
_rows_comp = MAX_ROWS_COMP or int(np.load(PRIMARY_Y, mmap_mode='r').shape[0])
# Terukur pada 70k baris cal, K=1000: fuzzy 46 s per kandidat x 15 = 11,6 mnt,
# rc3p 68 s, clustered 2 s -> ~13 mnt per seed. Puncak RSS 3,45 GB.
print('  perkiraan {:.0f} menit ({} baris, puncak RSS terukur ~3,5 GB)'.format(
    13.0 * len(SEEDS_COMP) * _rows_comp / 100000.0, _rows_comp))
print('  ' + drv.COMPETITOR_COST)
for s in SEEDS_COMP:
    run_one('C_comp', 'competitors', scores=PRIMARY_S, labels=PRIMARY_Y,
            dataset='ccc_imagenet', max_rows=MAX_ROWS_COMP, seed=s, alpha=ALPHA,
            frac_cal=FRAC_CAL_MAIN, n_cal=N_CAL_MAIN, competitors=True,
            head_weights=HEAD_W, head_bias=HEAD_B)
save_to_drive('fase 3 pesaing')

## FASE 4 — ImageNet-C

Kalibrasi pada citra **bersih**, evaluasi pada citra **terkorupsi**. Karena keduanya
dari distribusi berbeda, **exchangeability tidak berlaku dan jaminan konformal batal
untuk semua metode di sini, bukan hanya PCC** — dibaca sebagai uji ketahanan, dan
harus dijawab dengan literatur (weighted CP Tibshirani, adaptive CP Gibbs–Candès).
Lihat audit F4.

50 baris eval/kelas → regime A, jadi statistik per-kelas terukur di sini.

In [ ]:
print('FASE 4: ImageNet-C |', len(IC_CASES), 'kondisi')
print('  kalibrasi BERSIH -> evaluasi TERKORUPSI: exchangeability tidak berlaku')
N_CAL_IC, dropped_ic = n_cal_for(IC_CASES[0]['cal_size'] // 1000)
print('  {} baris kalibrasi/kelas -> n_cal={} (dibuang: {})'.format(
    IC_CASES[0]['cal_size'] // 1000, N_CAL_IC, dropped_ic))
for i, c in enumerate(IC_CASES, 1):
    if IC_SEVERITIES and c['severity'] not in IC_SEVERITIES:
        continue
    cs, cl = softmax_npy(c['cal'], c['T'], '{}/{}_s{}_cal'.format(
        IC_WORK, c['sub'], c['seed']))
    ts, tl = softmax_npy(c['test'], c['T'], '{}/{}_test'.format(IC_WORK, c['sub']))
    run_one('D_imagenetc', c['sub'], scores=cs, labels=cl,
            eval_scores=ts, eval_labels=tl, dataset='imagenetc_' + c['sub'],
            seed=c['seed'], alpha=ALPHA, frac_desc=0.0, frac_cal=1.0,
            n_cal=N_CAL_IC, competitors=COMP_ON_IC,
            head_weights=HEAD_W, head_bias=HEAD_B)
save_to_drive('fase 4 ImageNet-C')

## FASE 5 — backbone

Aturan keras `AGENTS.md`: baseline dihitung ulang pada backbone yang sama, tidak
pernah dibandingkan lintas backbone. Karena itu ketiganya dijalankan dengan protokol
identik dari dump 50.000 baris notebook 11 — bukan disatukan dengan cache 25k/25k
notebook 07.

In [ ]:
print('FASE 5: backbone |', sorted(BACKBONES))
for nm in sorted(BACKBONES):
    d = BACKBONES[nm]
    K = int(np.load(d + '/scores.npy', mmap_mode='r').shape[1])
    rows = int(np.load(d + '/labels.npy', mmap_mode='r').shape[0])
    n_cal_bb, dropped = n_cal_for(int(rows * FRAC_CAL_BB) // K)
    print('  {}: {} baris, {} kelas -> {} baris cal/kelas, n_cal={} (dibuang {})'.format(
        nm, rows, K, int(rows * FRAC_CAL_BB) // K, n_cal_bb, dropped))
    for s in SEEDS_MAIN:
        run_one('E_backbone', nm, scores=d + '/scores.npy', labels=d + '/labels.npy',
                dataset='backbone_' + nm, seed=s, alpha=ALPHA,
                frac_desc=0.0, frac_cal=FRAC_CAL_BB, n_cal=n_cal_bb,
                head_weights=d + '/fc_weight.npy', head_bias=d + '/fc_bias.npy')
save_to_drive('fase 5 backbone')

## FASE 6 — tabel akhir

In [ ]:
def agg(phase, tag=None):
    """Rerata + CI antar seed untuk satu lengan, pada statistik primer tabelnya."""
    got = [r for r in RESULTS if r['phase'] == phase and (tag is None or r['tag'] == tag)
           and r['res'].get('table_2_heldout')]
    if not got:
        return None
    out = {'n': len(got)}
    for lab, pick in (
            ('delta', lambda t: t['delta'].get(t['primary_stat'])),
            ('oracle', lambda t: t.get('delta_oracle', {}).get(t['primary_stat'])),
            ('marginal_cov', lambda t: t['pcc'].get('marginal_cov')),
            ('avg_set_size', lambda t: t['pcc'].get('avg_set_size')),
            ('sscv', lambda t: t['pcc'].get('sscv')),
            ('empty', lambda t: t['pcc'].get('frac_empty_sets')),
            ('frac_below', lambda t: t['pcc'].get('frac_classes_below_target'))):
        v = [pick(r['res']['table_2_heldout']) for r in got]
        v = [float(x) for x in v if x is not None and np.isfinite(x)]
        out[lab] = mean_ci(np.array(v, float)) if v else None
    out['stat'] = got[0]['res']['table_2_heldout']['primary_stat']
    return out

def line(label, a):
    if not a or not a['delta']:
        print('  {:30s} (tidak ada hasil)'.format(label))
        return
    d, o = a['delta'], a['oracle']
    pct = ('{:4.0f}% plafon'.format(100 * d['mean'] / o['mean'])
           if o and o['mean'] > 1e-6 else '  plafon <= 0')
    print('  {:30s} {:+.4f} [{:+.4f},{:+.4f}] n={:2d} {} | ukuran {:.3f}'
          ' | cak {:.4f} | kosong {:.3f}'
          .format(label, d['mean'], d['ci_low'], d['ci_high'], a['n'], pct,
                  a['avg_set_size']['mean'] if a['avg_set_size'] else float('nan'),
                  a['marginal_cov']['mean'] if a['marginal_cov'] else float('nan'),
                  a['empty']['mean'] if a['empty'] else float('nan')))

TABLES = {}
print('=' * 78)
print('TABEL 2 (kelas held-out, n_y = 0) -- statistik primer per baris')
print('=' * 78)
for lab, ph, tg in ([('skor ' + s, 'S_score', 'score_' + s) for s in SCORES]
                    + [('alpha ' + str(a), 'S_alpha', 'alpha{}'.format(a)) for a in ALPHAS]
                    + [('backbone ' + b, 'E_backbone', b) for b in sorted(BACKBONES)]
                    + [('pesaing (konfigurasi primer)', 'C_comp', 'competitors')]):
    a = agg(ph, tg)
    TABLES[lab] = a
    line(lab, a)

print()
print('=' * 78)
print('IMAGENET-C -- kalibrasi bersih, evaluasi terkorupsi')
print('=' * 78)
ic = collections.defaultdict(list)
for r in RESULTS:
    if r['phase'] != 'D_imagenetc' or not r['res'].get('table_2_heldout'):
        continue
    ic[r['tag']].append(r)
for tag in sorted(ic):
    a = agg('D_imagenetc', tag)
    TABLES['imagenetc ' + tag] = a
    line(tag, a)
if ic:
    allv = [r['res']['table_2_heldout']['delta'][
                r['res']['table_2_heldout']['primary_stat']]
            for rs in ic.values() for r in rs]
    ci = mean_ci(np.array(allv, float))
    print('  {:30s} {:+.4f} [{:+.4f},{:+.4f}] n={} kondisi'.format(
        'GABUNGAN semua korupsi', ci['mean'], ci['ci_low'], ci['ci_high'], len(allv)))
    TABLES['imagenetc_pooled'] = ci

print()
print('=' * 78)
print('PESAING TERBIT pada konfigurasi primer (n_y = 0)')
print('=' * 78)
COMP = {}
cr = [r for r in RESULTS if r['phase'] == 'C_comp' and r['res'].get('table_2_heldout')]
if not cr:
    print('  fase 3 tidak menghasilkan apa pun -- periksa LTC_DIR')
else:
    t0 = cr[0]['res']['table_2_heldout']
    st = t0['primary_stat']
    names = sorted(t0.get('delta_competitors', {}))
    print('  {:30s} {:>9s} {:>10s} {:>22s}'.format('metode', 'delta', 'ukuran',
                                                   'kelas tak terdefinisi'))
    dm = [r['res']['table_2_heldout']['delta'][st] for r in cr]
    print('  {:30s} {:>+9.4f} {:>10.3f} {:>22s}'.format(
        'PCC', float(np.mean(dm)),
        float(np.mean([r['res']['table_2_heldout']['pcc']['avg_set_size'] for r in cr])),
        '0 (phi tak butuh label)'))
    for nm in names:
        d = [r['res']['table_2_heldout']['delta_competitors'][nm].get(st) for r in cr]
        d = [x for x in d if x is not None and np.isfinite(x)]
        c0 = [r['res']['table_2_heldout']['competitors'][nm] for r in cr]
        und = float(np.mean([c['n_classes_undefined'] for c in c0]))
        COMP[nm] = {'delta_mean': float(np.mean(d)) if d else None,
                    'n': len(d),
                    'n_classes_undefined_mean': und,
                    'hyperparameter': c0[0].get('hyperparameter')}
        print('  {:30s} {:>+9.4f} {:>10.3f} {:>10.1f} / {:<4d}  {}'.format(
            nm, float(np.mean(d)) if d else float('nan'),
            float(np.mean([c['avg_set_size'] for c in c0])), und,
            t0['n_classes'], c0[0].get('hyperparameter', '')))
    print()
    print('  Cakupan worst-class ditentukan oleh kelas yang GAGAL ditangani sebuah')
    print('  metode, bukan yang berhasil -- kolom terakhir itu yang menjelaskan')
    print('  kenapa sebuah metode bisa memberi ambang informatif ke sebagian besar')
    print('  kelas held-out dan tetap skor tepat 0,0000.')

RUNTIME = {}
for r in RESULTS:
    RUNTIME.setdefault(r['phase'], []).append(r['secs'])
RUNTIME = {k: {'mean_s': float(np.mean(v)), 'n': len(v)} for k, v in RUNTIME.items()}
print()
for k, v in sorted(RUNTIME.items()):
    print('  {:14s} {:7.1f}s rata-rata x{}'.format(k, v['mean_s'], v['n']))

CAVEATS = [
    'ImageNet-C: kalibrasi BERSIH, evaluasi TERKORUPSI. Exchangeability tidak',
    '  berlaku, jadi jaminan konformal batal untuk SEMUA metode di sana, bukan',
    '  hanya PCC. Dibaca sebagai uji ketahanan, bukan jaminan (audit F4).',
    'Pesaing: bandwidth fuzzy disapu dan yang TERBAIK diambil (oracle-tuned untuk',
    '  pesaingnya, arah yang konservatif bagi kita); seri dipecahkan ke arah',
    '  kandidat yang meninggalkan lebih sedikit kelas tak terdefinisi.',
    'Pesaing hanya di konfigurasi primer: biayanya terukur ~25 mnt/run.',
    'Plafon oracle memakai label EVAL DUA kali (untuk delta dan untuk lambda),',
    '  jadi ia tak tercapai secara konstruksi -- plafon, bukan metode.',
    'n_cal diturunkan secara eksplisit bila irisan kalibrasi terlalu tipis, dan',
    '  nilai yang dibuang dicetak; ia kriteria pra-registrasi.',
]
for c in CAVEATS:
    print('CAVEAT:', c)

path = write_report('pcc/reports', '12_paper_tables',
                    hypothesis=drv.HYPOTHESIS, pass_criteria=drv.PASS_CRITERIA,
                    config={'alpha': ALPHA, 'alphas': list(ALPHAS),
                            'scores': list(SCORES), 'seeds_main': list(SEEDS_MAIN),
                            'seeds_comp': list(SEEDS_COMP),
                            'backbones': sorted(BACKBONES),
                            'n_imagenetc_conditions': len(IC_CASES),
                            'competitors_on_imagenetc': bool(COMP_ON_IC),
                            'seed': SEED},
                    seed=SEED,
                    results={'tables': TABLES, 'competitors': COMP,
                             'runtime': RUNTIME, 'caveats': CAVEATS,
                             'n_ok': len(RESULTS), 'n_failed': len(FAILED),
                             'failed': FAILED},
                    conclusion='SELESAI' if not FAILED else 'SEBAGIAN',
                    started_at=time.time())
print()
print('laporan:', path)
print('ok {} | gagal {}'.format(len(RESULTS), len(FAILED)))
save_to_drive('final')
z = shutil.make_archive(RUN_DIR, 'zip', RUN_DIR)
print('zip', z, '{:.1f} MB'.format(os.path.getsize(z) / 1e6))